## Joins with `PySpark` and `SQL` in Azure Databricks

### Inner Join - Orders with Restaurants

#### SQL Code Example

In [ ]:
%sql

SELECT 
  o.order_id,
  o.city,
  o.state,
  o.order_value_usd,
  r.restaurant_name,
  r.cuisine
FROM food_delivery_lab.bronze.orders o
INNER JOIN food_delivery_lab.bronze.restaurants r
  ON o.restaurant_id = r.restaurant_id;

#### PySpark Code Example

In [ ]:
from pyspark.sql.functions import col

orders = spark.table("food_delivery_lab.bronze.orders")
restaurants = spark.table("food_delivery_lab.bronze.restaurants")

joined_df = orders.alias("o").join(
    restaurants.alias("r"),
    col("o.restaurant_id") == col("r.restaurant_id"),
    "inner"
)

result = joined_df.select(
    "o.order_id",
    "o.city",
    "o.state",
    "o.order_value_usd",
    "r.restaurant_name",
    "r.cuisine"
)

display(result)

### Inner Join - Orders with Delivery Executives

#### SQL Code Example

In [ ]:
%sql

SELECT 
  o.order_id,
  o.city,
  o.order_value_usd,
  d.driver_name,
  d.vehicle_type,
  d.rating AS driver_rating
FROM food_delivery_lab.bronze.orders o
INNER JOIN food_delivery_lab.bronze.drivers d
  ON o.driver_id = d.driver_id;

#### PySpark Code Example

In [ ]:
drivers = spark.table("food_delivery_lab.bronze.drivers")

joined_df = orders.alias("o").join(
    drivers.alias("d"),
    col("o.driver_id") == col("d.driver_id"),
    "inner"
)

result = joined_df.select(
    "o.order_id",
    "o.city",
    "o.order_value_usd",
    "d.driver_name",
    "d.vehicle_type",
    col("d.rating").alias("driver_rating")
)

display(result)

### Left Join - Orders with Payments

#### SQL Code Example

In [ ]:
%sql

SELECT 
  o.order_id,
  o.order_value_usd,
  p.payment_status,
  p.payment_time
FROM food_delivery_lab.bronze.orders o
LEFT JOIN food_delivery_lab.bronze.payments p
  ON o.order_id = p.order_id;

#### PySpark Code Example

In [ ]:
payments = spark.table("food_delivery_lab.bronze.payments")

joined_df = orders.alias("o").join(
    payments.alias("p"),
    col("o.order_id") == col("p.order_id"),
    "left"
)

display(joined_df.select(
    "o.order_id",
    "o.order_value_usd",
    "p.payment_status",
    "p.payment_time"
))

### Left Anti Join - Orders without Payments

#### SQL Code Example

In [ ]:
%sql

SELECT 
  o.order_id,
  o.city,
  o.order_value_usd
FROM food_delivery_lab.bronze.orders o
LEFT JOIN food_delivery_lab.bronze.payments p
  ON o.order_id = p.order_id
WHERE p.payment_id IS NULL;

#### PySpark Code Example

In [ ]:
no_payment_df = orders.join(
    payments,
    "order_id",
    "left_anti"
)

display(no_payment_df.select("order_id", "city", "order_value_usd"))

### Multi-Table Join (Orders + Restaurants + Executives + Payments)

#### SQL Code Example

In [ ]:
%sql

SELECT 
  o.order_id,
  o.city,
  o.state,
  o.order_value_usd,
  r.restaurant_name,
  d.driver_name,
  p.payment_status
FROM food_delivery_lab.bronze.orders o
LEFT JOIN food_delivery_lab.bronze.restaurants r
  ON o.restaurant_id = r.restaurant_id
LEFT JOIN food_delivery_lab.bronze.drivers d
  ON o.driver_id = d.driver_id
LEFT JOIN food_delivery_lab.bronze.payments p
  ON o.order_id = p.order_id;

#### PySpark Code Example

In [ ]:
full_df = orders.alias("o") \
    .join(restaurants.alias("r"), col("o.restaurant_id") == col("r.restaurant_id"), "left") \
    .join(drivers.alias("d"), col("o.driver_id") == col("d.driver_id"), "left") \
    .join(payments.alias("p"), col("o.order_id") == col("p.order_id"), "left")

result = full_df.select(
    "o.order_id",
    "o.city",
    "o.state",
    "o.order_value_usd",
    "r.restaurant_name",
    "d.driver_name",
    "p.payment_status"
)

display(result)

### Join + Aggregation (Business Insight)

#### SQL Code Example - Revenue per restaurant

In [ ]:
%sql

SELECT 
  r.restaurant_name,
  SUM(o.order_value_usd) AS total_revenue
FROM food_delivery_lab.bronze.orders o
JOIN food_delivery_lab.bronze.restaurants r
  ON o.restaurant_id = r.restaurant_id
WHERE o.delivery_status = 'Delivered'
GROUP BY r.restaurant_name
ORDER BY total_revenue DESC;

#### PySpark Code Example - Revenue per restaurant

In [ ]:
from pyspark.sql.functions import sum

result = orders.alias("o") \
    .join(restaurants.alias("r"), "restaurant_id") \
    .filter(col("delivery_status") == "Delivered") \
    .groupBy("r.restaurant_name") \
    .agg(sum("order_value_usd").alias("total_revenue")) \
    .orderBy(col("total_revenue").desc())

display(result)